In [26]:
!pip install gensim networkx


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import numpy as np
import nltk
import re
from scipy import spatial
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
import networkx as nx

In [28]:
nltk.download('omw-1.4')
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
print(nltk.data.path)

['C:\\Users\\morga/nltk_data', 'c:\\Users\\morga\\anaconda3\\nltk_data', 'c:\\Users\\morga\\anaconda3\\share\\nltk_data', 'c:\\Users\\morga\\anaconda3\\lib\\nltk_data', 'C:\\Users\\morga\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data']


In [30]:
try:
    nltk.data.find("tokenizers/punkt")
    print("punkt installed")
except LookupError:
    print("punkt not installed")

punkt installed


In [31]:
text = """Mary had a little lamb,  little lamb, little lamb,  Mary had a little lamb, its fleece was white as snow. And everywhere that Mary went,  Mary went, Mary went,  and everywhere that Mary went, the lamb was sure to go. It followed her to school one day  school one day, school one day,  It followed her to school one day, which was against the rules. It made the children laugh and play,  laugh and play, laugh and play,  it made the children laugh and play to see And so the teacher turned it out,  turned it out, turned it out,  And so the teacher turned it out, but still it lingered near, And waited patiently about,  patiently about, patiently about,  And waited patiently about till Mary did appear. "Why does the lamb love Mary so?"  Love Mary so? Love Mary so?  "Why does the lamb love Mary so," the eager children cry. "Why, Mary loves the lamb, you know."  The lamb, you know, the lamb, you know, "Why, Mary loves the lamb, you know," the teacher did repl"""

In [32]:
sentences = sent_tokenize(text)
print(len(sentences))

10


In [33]:
sentNew = [re.sub(r"[^\w\s]","",s.lower()) for s in sentences] #Not: letters, number, underscore, space
print(sentNew)

['mary had a little lamb  little lamb little lamb  mary had a little lamb its fleece was white as snow', 'and everywhere that mary went  mary went mary went  and everywhere that mary went the lamb was sure to go', 'it followed her to school one day  school one day school one day  it followed her to school one day which was against the rules', 'it made the children laugh and play  laugh and play laugh and play  it made the children laugh and play to see and so the teacher turned it out  turned it out turned it out  and so the teacher turned it out but still it lingered near and waited patiently about  patiently about patiently about  and waited patiently about till mary did appear', 'why does the lamb love mary so', 'love mary so', 'love mary so', 'why does the lamb love mary so the eager children cry', 'why mary loves the lamb you know', 'the lamb you know the lamb you know why mary loves the lamb you know the teacher did repl']


In [34]:
stop_words = stopwords.words("english")
sentTokens = [[w for w in s.split() if w not in stop_words] for s in sentNew]
print(sentTokens)

[['mary', 'little', 'lamb', 'little', 'lamb', 'little', 'lamb', 'mary', 'little', 'lamb', 'fleece', 'white', 'snow'], ['everywhere', 'mary', 'went', 'mary', 'went', 'mary', 'went', 'everywhere', 'mary', 'went', 'lamb', 'sure', 'go'], ['followed', 'school', 'one', 'day', 'school', 'one', 'day', 'school', 'one', 'day', 'followed', 'school', 'one', 'day', 'rules'], ['made', 'children', 'laugh', 'play', 'laugh', 'play', 'laugh', 'play', 'made', 'children', 'laugh', 'play', 'see', 'teacher', 'turned', 'turned', 'turned', 'teacher', 'turned', 'still', 'lingered', 'near', 'waited', 'patiently', 'patiently', 'patiently', 'waited', 'patiently', 'till', 'mary', 'appear'], ['lamb', 'love', 'mary'], ['love', 'mary'], ['love', 'mary'], ['lamb', 'love', 'mary', 'eager', 'children', 'cry'], ['mary', 'loves', 'lamb', 'know'], ['lamb', 'know', 'lamb', 'know', 'mary', 'loves', 'lamb', 'know', 'teacher', 'repl']]


In [35]:
w2v = Word2Vec(sentTokens,
               vector_size=1,
               min_count=1,
               epochs=500)

In [36]:
sentEmbed = [[w2v.wv[w][0] for w in s] for s in sentTokens]
print(sentEmbed)

[[-1.6407374, -1.3596736, -1.442562, -1.3596736, -1.442562, -1.3596736, -1.442562, -1.6407374, -1.3596736, -1.442562, -1.0113249, -1.1525038, -0.899166], [-1.6193572, -1.6407374, -1.7321134, -1.6407374, -1.7321134, -1.6407374, -1.7321134, -1.6193572, -1.6407374, -1.7321134, -1.442562, -1.0356535, -0.98240215], [-1.7564175, -1.81608, -1.6482944, -1.5489225, -1.81608, -1.6482944, -1.5489225, -1.81608, -1.6482944, -1.5489225, -1.7564175, -1.81608, -1.6482944, -1.5489225, -1.2376314], [-1.5948995, -1.9882413, -1.8271258, -2.3203607, -1.8271258, -2.3203607, -1.8271258, -2.3203607, -1.5948995, -1.9882413, -1.8271258, -2.3203607, -1.5235313, -1.9157766, -2.237265, -2.237265, -2.237265, -1.9157766, -2.237265, -2.7020442, -2.148894, -2.3787003, -2.1507666, -2.68766, -2.68766, -2.68766, -2.1507666, -2.68766, -2.045791, -1.6407374, -1.7658983], [-1.442562, -0.88486165, -1.6407374], [-0.88486165, -1.6407374], [-0.88486165, -1.6407374], [-1.442562, -0.88486165, -1.6407374, -1.0942882, -1.9882413, -

In [37]:
maxLen = max([len(tokens) for tokens in sentEmbed])
print(maxLen)

31


In [38]:
sentEmbed = [np.pad(e,(0,maxLen-len(e)),"constant") for e in sentEmbed]
print(sentEmbed)

[array([-1.6407374, -1.3596736, -1.442562 , -1.3596736, -1.442562 ,
       -1.3596736, -1.442562 , -1.6407374, -1.3596736, -1.442562 ,
       -1.0113249, -1.1525038, -0.899166 ,  0.       ,  0.       ,
        0.       ,  0.       ,  0.       ,  0.       ,  0.       ,
        0.       ,  0.       ,  0.       ,  0.       ,  0.       ,
        0.       ,  0.       ,  0.       ,  0.       ,  0.       ,
        0.       ], dtype=float32), array([-1.6193572 , -1.6407374 , -1.7321134 , -1.6407374 , -1.7321134 ,
       -1.6407374 , -1.7321134 , -1.6193572 , -1.6407374 , -1.7321134 ,
       -1.442562  , -1.0356535 , -0.98240215,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ], dtype=float32), array([-1.7564175, -1.81608  , -1.6482944, -1.5489225, -1.81608  ,
       -1.6482944, -1.548

In [39]:
simMatrix = np.zeros((len(sentEmbed),len(sentEmbed)))
print(simMatrix.shape)

(10, 10)


In [40]:
for i,rowEmbed in enumerate(sentEmbed):
    for j, colEmbed in enumerate(sentEmbed):
        simMatrix[i][j] = 1-spatial.distance.cosine(rowEmbed,colEmbed)

print(simMatrix)

[[1.         0.9945026  0.93874538 0.58128047 0.51133937 0.40107122
  0.40107122 0.68505996 0.5855642  0.91054726]
 [0.9945026  1.         0.93583262 0.57929707 0.49619091 0.39036265
  0.39036265 0.68940628 0.5741474  0.91387868]
 [0.93874538 0.93583262 1.         0.63111717 0.45156425 0.37819174
  0.37819174 0.62096697 0.51518261 0.81042176]
 [0.58128047 0.57929707 0.63111717 1.         0.2501466  0.20944017
  0.20944017 0.3733663  0.31105071 0.4962475 ]
 [0.51133937 0.49619091 0.45156425 0.2501466  1.         0.62091643
  0.62091643 0.70096511 0.88321048 0.5395714 ]
 [0.40107122 0.39036265 0.37819174 0.20944017 0.62091643 1.
  1.         0.43524081 0.6007061  0.40473443]
 [0.40107122 0.39036265 0.37819174 0.20944017 0.62091643 1.
  1.         0.43524081 0.6007061  0.40473443]
 [0.68505996 0.68940628 0.62096697 0.3733663  0.70096511 0.43524081
  0.43524081 1.         0.76770443 0.74264699]
 [0.5855642  0.5741474  0.51518261 0.31105071 0.88321048 0.6007061
  0.6007061  0.76770443 1.   

In [41]:
graph = nx.from_numpy_array(simMatrix)

In [42]:
ranks = nx.pagerank(graph)
print(ranks)

{0: 0.11087699569621616, 1: 0.11024736026383694, 2: 0.10616483738062318, 3: 0.07898366587943162, 4: 0.098397066989368, 5: 0.09006793213143763, 6: 0.09006793213143763, 7: 0.10330248136332112, 8: 0.10345853367267122, 9: 0.10843319449165649}


In [43]:
sentScores = {sent:ranks[index] for index,sent in enumerate(sentences)}
print(sentScores)

{'Mary had a little lamb,  little lamb, little lamb,  Mary had a little lamb, its fleece was white as snow.': 0.11087699569621616, 'And everywhere that Mary went,  Mary went, Mary went,  and everywhere that Mary went, the lamb was sure to go.': 0.11024736026383694, 'It followed her to school one day  school one day, school one day,  It followed her to school one day, which was against the rules.': 0.10616483738062318, 'It made the children laugh and play,  laugh and play, laugh and play,  it made the children laugh and play to see And so the teacher turned it out,  turned it out, turned it out,  And so the teacher turned it out, but still it lingered near, And waited patiently about,  patiently about, patiently about,  And waited patiently about till Mary did appear.': 0.07898366587943162, '"Why does the lamb love Mary so?"': 0.098397066989368, 'Love Mary so?': 0.09006793213143763, '"Why does the lamb love Mary so," the eager children cry.': 0.10330248136332112, '"Why, Mary loves the l

In [44]:
top4 = sorted(sentScores.items(),key=lambda x:x[1],reverse=True)[0:4]
print(top4)

[('Mary had a little lamb,  little lamb, little lamb,  Mary had a little lamb, its fleece was white as snow.', 0.11087699569621616), ('And everywhere that Mary went,  Mary went, Mary went,  and everywhere that Mary went, the lamb was sure to go.', 0.11024736026383694), ('The lamb, you know, the lamb, you know, "Why, Mary loves the lamb, you know," the teacher did repl', 0.10843319449165649), ('It followed her to school one day  school one day, school one day,  It followed her to school one day, which was against the rules.', 0.10616483738062318)]


In [45]:
for sent in top4:
    print(sent[0])

Mary had a little lamb,  little lamb, little lamb,  Mary had a little lamb, its fleece was white as snow.
And everywhere that Mary went,  Mary went, Mary went,  and everywhere that Mary went, the lamb was sure to go.
The lamb, you know, the lamb, you know, "Why, Mary loves the lamb, you know," the teacher did repl
It followed her to school one day  school one day, school one day,  It followed her to school one day, which was against the rules.
